# Telco Customer Churn – Data Preprocessing & Feature Engineering

End-to-end preprocessing flow in one notebook: from the raw `telco.csv` to the final train/test split
saved in the `SplitData` folder.

**Flow:** Load → Clean → Remove leakage → Select features → Engineer features → Split → Encode & scale → Save

## 1. Setup
Libraries used throughout the notebook and the input/output locations.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RAW_PATH = "../Dataset/telco.csv"
OUT_DIR = "../SplitData"
TARGET = "Churn Label"
RANDOM_STATE = 42

os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load the Raw Dataset
The original Telco dataset: one row per customer, 50 columns.

In [2]:
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
df.head()

Shape: (7043, 50)


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


## 3. Data Types
Confirm every column was read with the correct type (numeric columns such as `Total Charges` must be numeric).

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   str    
 1   Gender                             7043 non-null   str    
 2   Age                                7043 non-null   int64  
 3   Under 30                           7043 non-null   str    
 4   Senior Citizen                     7043 non-null   str    
 5   Married                            7043 non-null   str    
 6   Dependents                         7043 non-null   str    
 7   Number of Dependents               7043 non-null   int64  
 8   Country                            7043 non-null   str    
 9   State                              7043 non-null   str    
 10  City                               7043 non-null   str    
 11  Zip Code                           7043 non-null   int64  
 12  Lat

## 4. Missing Values
Identify which columns contain missing values before deciding how to treat each one.

In [4]:
missing = df.isnull().sum()
print(missing[missing > 0])

Offer             3877
Internet Type     1526
Churn Category    5174
Churn Reason      5174
dtype: int64


### 4.1 Offer
A missing `Offer` means the customer did not accept any promotion, so it is filled with `No Offer`
rather than imputed with the most common offer.

In [5]:
print(df["Offer"].value_counts(dropna=False), "\n")

df["Offer"] = df["Offer"].fillna("No Offer")
print(df["Offer"].value_counts(dropna=False))

Offer
NaN        3877
Offer B     824
Offer E     805
Offer D     602
Offer A     520
Offer C     415
Name: count, dtype: int64 

Offer
No Offer    3877
Offer B      824
Offer E      805
Offer D      602
Offer A      520
Offer C      415
Name: count, dtype: int64


### 4.2 Internet Type
Cross-checking with `Internet Service` shows every missing `Internet Type` belongs to a customer without
internet, so the value is structurally missing and is filled with `No Internet Service`.

In [6]:
print(df.groupby("Internet Service")["Internet Type"].apply(lambda s: s.isnull().sum()), "\n")

df["Internet Type"] = df["Internet Type"].fillna("No Internet Service")
print(df["Internet Type"].value_counts(dropna=False))

Internet Service
No     1526
Yes       0
Name: Internet Type, dtype: int64 

Internet Type
Fiber Optic            3035
DSL                    1652
No Internet Service    1526
Cable                   830
Name: count, dtype: int64


### 4.3 Remaining missing values
`Churn Category` and `Churn Reason` are only filled for customers who already churned. They are not
imputed because they are removed as leakage in Section 7.

In [7]:
missing = df.isnull().sum()
print(missing[missing > 0])

Churn Category    5174
Churn Reason      5174
dtype: int64


## 5. Duplicate and Invalid Records
Check for duplicated rows and for impossible values (negative charges or refunds).

In [8]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Customer IDs:", df["Customer ID"].duplicated().sum(), "\n")

financial_cols = ["Monthly Charge", "Total Charges", "Total Refunds",
                  "Total Extra Data Charges", "Total Long Distance Charges"]

for col in financial_cols:
    print(f"{col:<30} negative values: {(df[col] < 0).sum()}")

Duplicate rows: 0
Duplicate Customer IDs: 0 

Monthly Charge                 negative values: 0
Total Charges                  negative values: 0
Total Refunds                  negative values: 0
Total Extra Data Charges       negative values: 0
Total Long Distance Charges    negative values: 0


No duplicates and no invalid values were found, so no rows are removed.

## 6. Outliers (IQR Method)
Numerical columns are checked with the 1.5 × IQR rule.

In [9]:
numerical_cols = ["Age", "Number of Dependents", "Number of Referrals", "Tenure in Months",
                  "Monthly Charge", "Total Charges", "Total Refunds", "Total Extra Data Charges",
                  "Total Long Distance Charges", "Population", "CLTV"]

rows = []
for col in numerical_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    rows.append((col, n_out, round(n_out / len(df) * 100, 2)))

pd.DataFrame(rows, columns=["Column", "Outliers", "Percent"])

,Column,Outliers,Percent
0,Age,0,0.00
1,Number of Dependents,1627,23.10
2,Number of Referrals,676,9.60
3,Tenure in Months,0,0.00
4,Monthly Charge,0,0.00
5,Total Charges,0,0.00
6,Total Refunds,525,7.45
7,Total Extra Data Charges,728,10.34
8,Total Long Distance Charges,196,2.78
9,Population,57,0.81


**Decision – outliers are kept.** The flagged columns (dependents, referrals, refunds, extra data charges)
are naturally right-skewed: most customers have 0 and a few have higher values. These are real customer
behaviours, not data errors, so removing or capping them would discard valid information.

## 7. Target Variable and Data Leakage
`Churn Label` (`Yes` / `No`) is the target. Any column that is only known *after* a customer churns, or
that encodes the target directly, must be removed so the model only learns from pre-churn information.

In [10]:
y_check = (df[TARGET] == "Yes").astype(int)   # numeric copy used only for the checks below

print(df[TARGET].value_counts(), "\n")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64 

Churn Label
No     73.46
Yes    26.54
Name: proportion, dtype: float64


### 7.1 Customer Status

In [11]:
pd.crosstab(df["Customer Status"], df[TARGET])

Churn Label,No,Yes
Customer Status,,
Churned,0,1869
Joined,454,0
Stayed,4720,0


### 7.2 Churn Category / Churn Reason

In [12]:
for col in ["Churn Category", "Churn Reason"]:
    print(f"{col}: missing = {df[col].isnull().sum()}, "
          f"missing among non-churners = {df.loc[df[TARGET] == 'No', col].isnull().sum()}")

Churn Category: missing = 5174, missing among non-churners = 5174
Churn Reason: missing = 5174, missing among non-churners = 5174


### 7.3 Churn Score and Satisfaction Score

In [13]:
print("Churn Score vs target r =", round(np.corrcoef(df["Churn Score"], y_check)[0, 1], 3))
print(df.groupby(TARGET)["Churn Score"].agg(["mean", "min", "max"]).round(2), "\n")

ct = pd.crosstab(df["Satisfaction Score"], df[TARGET])
ct["churn_rate_%"] = (ct["Yes"] / ct.sum(axis=1) * 100).round(1)
print(ct)
print("\nSatisfaction Score vs target r =", round(np.corrcoef(df["Satisfaction Score"], y_check)[0, 1], 3))

Churn Score vs target r = 0.661
              mean  min  max
Churn Label                 
No           50.10    5   80
Yes          81.78   65   96 

Churn Label           No  Yes  churn_rate_%
Satisfaction Score                         
1                      0  922         100.0
2                      0  518         100.0
3                   2236  429          16.1
4                   1789    0           0.0
5                   1149    0           0.0

Satisfaction Score vs target r = -0.755


### 7.4 Leakage decisions

| Column | Reason for removal |
|---|---|
| `Customer ID` | Unique identifier – no predictive meaning. |
| `Customer Status` | `Churned` = all Yes, `Joined/Stayed` = all No – it is the target under another name. |
| `Churn Category`, `Churn Reason` | Only exist for customers who already churned. |
| `Churn Score` | Risk score derived from churn information, not a raw attribute. |
| `Satisfaction Score` | Scores 1–2 are 100% churn and 4–5 are 0% churn – near-perfect separation of the classes. |

In [14]:
leakage_cols = ["Customer ID", "Customer Status", "Churn Category",
                "Churn Reason", "Churn Score", "Satisfaction Score"]

## 8. Feature Selection

### 8.1 Zero-variance columns
A column with a single value cannot separate churners from non-churners.

In [15]:
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]

for c in constant_cols:
    print(f"{c}: only value -> {df[c].iloc[0]}")

Country: only value -> United States


State: only value -> California
Quarter: only value -> Q3


### 8.2 Geographic columns
Checked for cardinality (one-hot width) and relationship with churn.

In [16]:
geo_cols = ["City", "Zip Code", "Latitude", "Longitude", "Population"]

for c in geo_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        r = round(np.corrcoef(df[c], y_check)[0, 1], 3)
    else:
        r = "n/a (text)"
    print(f"{c:<12} unique: {df[c].nunique():>5}   r with target: {r}")

City         unique:  1106   r with target: n/a (text)
Zip Code     unique:  1626   r with target: -0.016
Latitude     unique:  1626   r with target: -0.042
Longitude    unique:  1625   r with target: 0.024
Population   unique:  1569   r with target: 0.052


**Decision – remove all geographic columns.** `City` and `Zip Code` have over 1,000 categories each
(would explode the feature space), all numeric location columns have |r| ≤ 0.06 with churn, and
`Zip Code`, `Latitude` and `Longitude` describe the same location.

### 8.3 Redundant columns
Several columns are exact recodes of another column. Only the more informative version is kept.

In [17]:
print(pd.crosstab(df["Under 30"], df["Age"] < 30), "\n")
print(pd.crosstab(df["Senior Citizen"], df["Age"] >= 65), "\n")
print(pd.crosstab(df["Dependents"], df["Number of Dependents"] > 0), "\n")
print(pd.crosstab(df["Referred a Friend"], df["Number of Referrals"] > 0), "\n")
print(pd.crosstab(df["Internet Service"], df["Internet Type"]))

Age       False  True 
Under 30              
No         5642      0
Yes           0   1401 



Age             False  True 
Senior Citizen              
No               5901      0
Yes                 0   1142 

Number of Dependents  False  True 
Dependents                        
No                     5416      0
Yes                       0   1627 

Number of Referrals  False  True 
Referred a Friend                
No                    3821      0
Yes                      0   3222 

Internet Type     Cable   DSL  Fiber Optic  No Internet Service
Internet Service                                               
No                    0     0            0                 1526
Yes                 830  1652         3035                    0


In [18]:
# Total Revenue is an accounting identity of the other financial columns
recomputed = (df["Total Charges"] - df["Total Refunds"]
              + df["Total Extra Data Charges"] + df["Total Long Distance Charges"])
print("Max |Total Revenue - recomputed|:", (recomputed - df["Total Revenue"]).abs().max(), "\n")

# Strongly correlated numeric pairs
corr = df.select_dtypes(include=np.number).drop(columns=["Churn Score", "Satisfaction Score"]).corr()
for i, a in enumerate(corr.columns):
    for b in corr.columns[i + 1:]:
        if abs(corr.loc[a, b]) > 0.7:
            print(f"{a:<30} {b:<30} r = {corr.loc[a, b]:.3f}")

Max |Total Revenue - recomputed|:

 1.8189894035458565e-12 

Zip Code                       Latitude                       r = 0.895
Zip Code                       Longitude                      r = -0.791
Latitude                       Longitude                      r = -0.886
Tenure in Months               Total Charges                  r = 0.826
Tenure in Months               Total Revenue                  r = 0.853
Total Charges                  Total Revenue                  r = 0.972
Total Long Distance Charges    Total Revenue                  r = 0.779


| Removed | Kept instead | Reason |
|---|---|---|
| `Under 30`, `Senior Citizen` | `Age` | Exact thresholds of `Age`. |
| `Dependents` | `Number of Dependents` | Exactly `Number of Dependents > 0`. |
| `Referred a Friend` | `Number of Referrals` | Exactly `Number of Referrals > 0`. |
| `Internet Service` | `Internet Type` | `Internet Type` already contains `No Internet Service`. |
| `Total Revenue` | individual charge columns | Exact sum of the other financial columns. |
| `Total Charges` | `Tenure in Months` + `Monthly Charge` | ≈ tenure × monthly charge (r = 0.83 with tenure). |
| `Total Long Distance Charges` | `Avg Monthly Long Distance Charges` | Total mostly repeats tenure; the average reflects usage. |

In [19]:
redundant_cols = ["Under 30", "Senior Citizen", "Dependents", "Referred a Friend", "Internet Service",
                  "Total Charges", "Total Revenue", "Total Long Distance Charges"]

## 9. Feature Engineering
Two business-driven features are created and checked against churn before being kept. Both are
computed row by row, so creating them before the split does not leak test information.

### 9.1 Total Services
Number of services the customer subscribes to (bundle depth).

In [20]:
service_cols = ["Phone Service", "Multiple Lines", "Internet Service", "Online Security", "Online Backup",
                "Device Protection Plan", "Premium Tech Support", "Streaming TV", "Streaming Movies",
                "Streaming Music", "Unlimited Data"]

df["Total Services"] = (df[service_cols] == "Yes").sum(axis=1)

svc = pd.crosstab(df["Total Services"], df[TARGET])
svc["churn_rate_%"] = (svc["Yes"] / svc.sum(axis=1) * 100).round(1)
svc

Churn Label,No,Yes,churn_rate_%
Total Services,,,
1,1086,105,8.8
2,407,74,15.4
3,274,242,46.9
4,436,322,42.5
5,507,285,36.0
6,522,278,34.8
7,525,229,30.4
8,565,178,24.0
9,410,115,21.9


Churn is low for 1 service, peaks around 3 services and falls again for heavily bundled customers.
This non-linear pattern is not visible in any single service flag, so the feature is kept.

### 9.2 Tenure Group
Tenure grouped into customer lifecycle stages.

In [21]:
df["Tenure Group"] = pd.cut(df["Tenure in Months"], bins=[0, 12, 24, 48, 72],
                            labels=["0-12", "13-24", "25-48", "49-72"], include_lowest=True)

tg = pd.crosstab(df["Tenure Group"], df[TARGET])
tg["churn_rate_%"] = (tg["Yes"] / tg.sum(axis=1) * 100).round(1)
tg

Churn Label,No,Yes,churn_rate_%
Tenure Group,,,
0-12,1149,1037,47.4
13-24,730,294,28.7
25-48,1269,325,20.4
49-72,2026,213,9.5


Churn drops from ~47% in the first year to ~10% after four years. The grouped version lets linear models
give each stage its own weight instead of assuming a straight-line tenure effect.

## 10. Final Feature Set
Apply all removal decisions and build the feature matrix `X` and target `y`.

In [22]:
removed_cols = leakage_cols + constant_cols + geo_cols + redundant_cols

removal_log = pd.DataFrame(
    [(c, "Identifier / leakage") for c in leakage_cols]
    + [(c, "Zero variance") for c in constant_cols]
    + [(c, "Geographic") for c in geo_cols]
    + [(c, "Redundant") for c in redundant_cols],
    columns=["Removed column", "Reason"]
)
print(removal_log.to_string(index=False))

             Removed column               Reason
                Customer ID Identifier / leakage
            Customer Status Identifier / leakage
             Churn Category Identifier / leakage
               Churn Reason Identifier / leakage
                Churn Score Identifier / leakage
         Satisfaction Score Identifier / leakage
                    Country        Zero variance
                      State        Zero variance
                    Quarter        Zero variance
                       City           Geographic
                   Zip Code           Geographic
                   Latitude           Geographic
                  Longitude           Geographic
                 Population           Geographic
                   Under 30            Redundant
             Senior Citizen            Redundant
                 Dependents            Redundant
          Referred a Friend            Redundant
           Internet Service            Redundant
              Total 

In [23]:
final_df = df.drop(columns=removed_cols)

X = final_df.drop(columns=[TARGET])
y = final_df[TARGET].map({"No": 0, "Yes": 1})

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("X:", X.shape, "| y:", y.shape)
print(f"\n{len(numeric_features)} numeric features:", numeric_features)
print(f"\n{len(categorical_features)} categorical features:", categorical_features)

X: (7043, 29) | y: (7043,)

11 numeric features: ['Age', 'Number of Dependents', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Refunds', 'Total Extra Data Charges', 'CLTV', 'Total Services']

18 categorical features: ['Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Group']


In [24]:
assert not any(c in X.columns for c in leakage_cols), "Leakage column found in X"
assert X.isnull().sum().sum() == 0, "X contains missing values"
print("Checks passed – no leakage columns, no missing values.")

Checks passed – no leakage columns, no missing values.


## 11. Train / Test Split
80 / 20 split, stratified on the target so both sets keep the same churn ratio (~26.5%).
The split is done **before** any encoding or scaling so the test set stays unseen.

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("\nTrain churn %:", (y_train.value_counts(normalize=True) * 100).round(2).to_dict())
print("Test churn %: ", (y_test.value_counts(normalize=True) * 100).round(2).to_dict())

X_train:

 (5634, 29) | X_test: (1409, 29)

Train churn %: {0: 73.46, 1: 26.54}
Test churn %:  {0: 73.46, 1: 26.54}


## 12. Encoding and Scaling Pipeline
- **Numerical:** median imputation + `StandardScaler` (needed for distance/gradient-based models such as
  Logistic Regression, KNN and SVM).
- **Categorical:** most-frequent imputation + `OneHotEncoder` (no false ordering between categories;
  `handle_unknown="ignore"` protects against unseen categories).

The imputers change nothing on this data but keep the pipeline safe for new records.

In [26]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 13. Fit on Training Data Only
The pipeline learns means, standard deviations and categories from the training set only, then the
same fitted rules are applied to the test set – this prevents information leakage.

In [27]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

X_train_processed: (5634, 57)
X_test_processed : (1409, 57)


In [28]:
# Scaled numeric columns in the training set should have mean ≈ 0 and std ≈ 1
n_num = len(numeric_features)
pd.DataFrame({
    "mean": X_train_processed[:, :n_num].mean(axis=0).round(3),
    "std": X_train_processed[:, :n_num].std(axis=0).round(3)
}, index=numeric_features)

,mean,std
Age,0.0,1.0
Number of Dependents,0.0,1.0
Number of Referrals,0.0,1.0
Tenure in Months,-0.0,1.0
Avg Monthly Long Distance Charges,-0.0,1.0
Avg Monthly GB Download,0.0,1.0
Monthly Charge,0.0,1.0
Total Refunds,0.0,1.0
Total Extra Data Charges,-0.0,1.0
CLTV,-0.0,1.0


## 14. Save the Final Split Data
Saved to the `SplitData` folder:

| File | Content |
|---|---|
| `X_train.csv`, `X_test.csv` | Encoded and scaled features |
| `y_train.csv`, `y_test.csv` | Target (0 = No churn, 1 = Churn) |
| `preprocessing_pipeline.pkl` | Fitted pipeline, reusable for new data |

In [29]:
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

X_train_df.to_csv(f"{OUT_DIR}/X_train.csv", index=False)
X_test_df.to_csv(f"{OUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUT_DIR}/y_test.csv", index=False)
joblib.dump(preprocessor, f"{OUT_DIR}/preprocessing_pipeline.pkl")

for f in sorted(os.listdir(OUT_DIR)):
    print(f)

X_test.csv
X_train.csv
preprocessing_pipeline.pkl
y_test.csv
y_train.csv


## 15. Verify the Saved Files
Reload the saved files to confirm shapes and class balance.

In [30]:
X_train_chk = pd.read_csv(f"{OUT_DIR}/X_train.csv")
X_test_chk = pd.read_csv(f"{OUT_DIR}/X_test.csv")
y_train_chk = pd.read_csv(f"{OUT_DIR}/y_train.csv").squeeze()
y_test_chk = pd.read_csv(f"{OUT_DIR}/y_test.csv").squeeze()

print("X_train:", X_train_chk.shape, "| y_train:", y_train_chk.shape)
print("X_test :", X_test_chk.shape, "| y_test :", y_test_chk.shape)
print("\nTrain target:", y_train_chk.value_counts().to_dict())
print("Test target: ", y_test_chk.value_counts().to_dict())

pipeline = joblib.load(f"{OUT_DIR}/preprocessing_pipeline.pkl")
print("\nPipeline reloaded:", type(pipeline).__name__)

X_train: (5634, 57) | y_train: (5634,)
X_test : (1409, 57) | y_test : (1409,)

Train target: {0: 4139, 1: 1495}
Test target:  {0: 1035, 1: 374}

Pipeline reloaded: ColumnTransformer


## 16. Summary

| Step | Decision |
|---|---|
| Missing values | `Offer` → `No Offer`; `Internet Type` → `No Internet Service` (structural missingness) |
| Duplicates / invalid | None found – no rows removed |
| Outliers | Kept – natural skew, not errors |
| Leakage | Removed ID, Customer Status, Churn Category/Reason/Score, Satisfaction Score |
| Feature selection | Removed zero-variance, geographic and redundant columns (22 removed in total) |
| Feature engineering | Added `Total Services` and `Tenure Group` |
| Split | 80/20 stratified, done before encoding/scaling |
| Encoding / scaling | One-hot for categorical, StandardScaler for numeric, fitted on train only |
| Output | 29 input features → 57 model-ready columns, saved in `SplitData/` |